<a href="https://colab.research.google.com/github/Shajs23224224/FFAIAssistant/blob/main/colab/FFAI_Colab_Notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎮 Free Fire AI - Colab + Drive + Flask + Ngrok

**Arquitectura mejorada:**
- 🧠 Colab = Motor IA (GPU)
- 💾 Drive = Persistencia (modelos, logs)
- 🌐 Flask API = Interfaz REST/WebSocket
- 🔗 Ngrok = URL pública
- 📱 Cliente = APK Android

## Instrucciones:
1. **Runtime → Run all** (Ctrl+F9)
2. Espera la URL de ngrok
3. Copia la URL en ServerConfig.kt del APK
4. ¡Juega!

## Celda 1: Montar Google Drive

In [39]:
from google.colab import drive
drive.mount('/content/drive')
print("✅ Drive montado en /content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Drive montado en /content/drive


## Celda 2: Instalar Dependencias

In [40]:
!pip install -q flask flask-socketio flask-cors pyngrok torch torchvision pillow numpy opencv-python-headless
print("✅ Dependencias instaladas")

✅ Dependencias instaladas


## Celda 3: Crear Módulos del Sistema

In [41]:
# Crear estructura de directorios
import os
os.makedirs('/content/ffai_core', exist_ok=True)

# Guardar engine.py
engine_code = '''"""Motor de IA"""

import torch
import torch.nn as nn
import numpy as np
import cv2
from PIL import Image
import io
import base64
import time
from typing import Dict, Tuple, List


class TacticalNN(nn.Module):
    def __init__(self, hidden=256, actions=10):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(128, hidden), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(hidden, hidden), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(hidden, 128), nn.ReLU(),
            nn.Linear(128, actions)
        )

    def forward(self, x):
        return torch.softmax(self.net(x), dim=-1)


class FFAIEngine:
    ACTIONS = {
        0: ("HOLD", 960, 700, 100),
        1: ("SHOOT", 1150, 750, 100),
        2: ("AIM", 1150, 600, 200),
        3: ("FORWARD", 960, 500, 300),
        4: ("BACK", 960, 900, 300),
        5: ("LEFT", 700, 700, 300),
        6: ("RIGHT", 1220, 700, 300),
        7: ("RELOAD", 1200, 900, 1500),
        8: ("HEAL", 150, 800, 500),
        9: ("JUMP", 960, 700, 100),
    }

    def __init__(self, device="cpu"):
        self.device = torch.device(device)
        self.model = TacticalNN().to(self.device)
        self.model.eval()
        self.frame_count = 0
        self.session_start = time.time()
        print(f"🧠 Engine listo en {device}")

    def decode_image(self, b64):
        try:
            img_bytes = base64.b64decode(b64)
            img = Image.open(io.BytesIO(img_bytes))
            return np.array(img)
        except:
            return np.zeros((135, 240, 3), dtype=np.uint8)

    def analyze_frame(self, img):
        h, w = img.shape[:2]
        health_zone = img[int(h*0.88):int(h*0.96), int(w*0.02):int(w*0.22)]
        ammo_zone = img[int(h*0.88):int(h*0.96), int(w*0.78):int(w*0.98)]
        enemy_zone = img[int(h*0.15):int(h*0.55), int(w*0.35):int(w*0.65)]

        hsv = cv2.cvtColor(health_zone, cv2.COLOR_RGB2HSV)
        green = cv2.inRange(hsv, np.array([35,40,40]), np.array([85,255,255]))
        health = min(np.sum(green>0)/green.size*3, 1.0)

        gray = cv2.cvtColor(ammo_zone, cv2.COLOR_RGB2GRAY)
        ammo = min(np.sum(gray>200)/gray.size*3, 1.0)

        hsv = cv2.cvtColor(enemy_zone, cv2.COLOR_RGB2HSV)
        r1 = cv2.inRange(hsv, np.array([0,50,50]), np.array([10,255,255]))
        r2 = cv2.inRange(hsv, np.array([160,50,50]), np.array([180,255,255]))
        enemy = (np.sum(cv2.bitwise_or(r1,r2)>0) / hsv.size) > 0.08

        return {"health": float(health), "ammo": float(ammo), "enemy": bool(enemy)}

    def process(self, image_b64, client_state=None):
        self.frame_count += 1
        img = self.decode_image(image_b64)
        state = self.analyze_frame(img)

        # Heurísticas
        if state["health"] < 0.25:
            return {"action": "HEAL", "x": 150, "y": 800, "duration": 500, "state": state}
        if state["enemy"] and state["ammo"] < 0.1:
            return {"action": "RELOAD", "x": 1200, "y": 900, "duration": 1500, "state": state}
        if state["enemy"]:
            return {"action": "SHOOT", "x": 1150, "y": 750, "duration": 100, "state": state}

        # IA
        vec = torch.FloatTensor([state["health"], state["ammo"], float(state["enemy"]), 1.0, 0.5]).to(self.device)
        with torch.no_grad():
            probs = self.model(torch.cat([vec, torch.zeros(123).to(self.device)]))
            action_idx = torch.multinomial(probs, 1).item()

        name, x, y, dur = self.ACTIONS.get(action_idx, ("HOLD", 960, 700, 100))
        return {"action": name, "x": x, "y": y, "duration": dur, "state": state}
'''

with open('/content/ffai_core/engine.py', 'w') as f:
    f.write(engine_code)

print("✅ engine.py creado")

✅ engine.py creado


## Celda 4: Configurar GPU

In [42]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cuda":
    print(f"🚀 GPU Tesla T4 activa")
    print(f"🔥 Memoria: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
else:
    print("⚠️ Usando CPU (más lento)")
    print("💡 Runtime → Change runtime type → GPU")

🚀 GPU Tesla T4 activa
🔥 Memoria: 15.6 GB


## Celda 5: Inicializar Motor IA

In [43]:
from ffai_core.engine import FFAIEngine

# Crear motor
engine = FFAIEngine(device=device)

# Intentar cargar modelo desde Drive si existe
import os
model_path = "/content/drive/MyDrive/FFAI/models/ff_model.pt"
if os.path.exists(model_path):
    print(f"📂 Cargando modelo desde Drive...")
    # engine.load_model(model_path)  # Implementar si hay modelos guardados
else:
    print(f"💾 El modelo se guardará en: {model_path}")
    os.makedirs(os.path.dirname(model_path), exist_ok=True)

print("✅ Motor IA listo")

🧠 Engine listo en cuda
💾 El modelo se guardará en: /content/drive/MyDrive/FFAI/models/ff_model.pt
✅ Motor IA listo


## Celda 6: Crear API Flask

In [44]:
from flask import Flask, request, jsonify
from flask_socketio import SocketIO, emit, join_room, leave_room
from flask_cors import CORS
import json
import time

app = Flask(__name__)
CORS(app)
socketio = SocketIO(app, cors_allowed_origins="*", async_mode="threading")

# Stats
stats = {
    "started": time.time(),
    "frames_processed": 0,
    "clients_connected": 0,
    "room_count": 0
}

# ============================================
# HTTP ROUTES
# ============================================

@app.route("/")
def root():
    return jsonify({
        "status": "🟢 Online",
        "service": "FFAI API",
        "version": "3.1.0",
        "device": device,
        "socketio_namespace": "/ffai",
        "stats": {
            "frames": stats["frames_processed"],
            "clients": stats["clients_connected"],
            "uptime": int(time.time() - stats["started"])
        }
    })

@app.route("/health")
def health():
    return jsonify({
        "status": "ok",
        "socketio": "available",
        "namespace": "/ffai"
    })

# ============================================
# SOCKETIO EVENTS - NAMESPACE /ffai
# ============================================

@socketio.on("connect", namespace="/ffai")
def handle_ffai_connect():
    stats["clients_connected"] += 1
    stats["room_count"] += 1

    # Crear room única para este cliente
    client_room = f"client_{request.sid}"
    join_room(client_room)

    print(f"📱 [/ffai] Cliente conectado: {request.sid}")
    print(f"   Total clientes: {stats['clients_connected']}")

    emit("connected", {
        "status": "ok",
        "client_id": request.sid,
        "room": client_room
    })

@socketio.on("disconnect", namespace="/ffai")
def handle_ffai_disconnect():
    stats["clients_connected"] = max(0, stats["clients_connected"] - 1)
    stats["room_count"] = max(0, stats["room_count"] - 1)
    print(f"📴 [/ffai] Cliente desconectado: {request.sid}")

@socketio.on("frame", namespace="/ffai")
def handle_ffai_frame(data):
    try:
        start_time = time.time()

        # Parsear data si es string
        if isinstance(data, str):
            data = json.loads(data)

        # Extraer frame y metadata
        image_b64 = data.get("imageBase64", "")
        client_state = data.get("clientState", {})
        frame_quality = data.get("quality", 50)

        # Procesar con IA
        result = engine.process(image_b64, client_state)

        # Actualizar stats
        stats["frames_processed"] += 1

        # Calcular latency
        processing_time = (time.time() - start_time) * 1000  # ms

        # Enviar respuesta SOLO a este cliente (usando su room)
        client_room = f"client_{request.sid}"
        emit("action", {
            "type": "action",
            "action": result["action"],
            "coordinates": {"x": result["x"], "y": result["y"]},
            "duration": result["duration"],
            "confidence": 0.85,  # Placeholder
            "game_state": result["state"],
            "server_latency_ms": processing_time
        }, room=client_room)

        # Log cada 100 frames
        if stats["frames_processed"] % 100 == 0:
            print(f"📊 Frames procesados: {stats['frames_processed']} | "
                  f"Clientes: {stats['clients_connected']} | "
                  f"Avg latency: {processing_time:.1f}ms")

    except Exception as e:
        print(f"❌ Error procesando frame: {e}")
        emit("error", {
            "message": str(e),
            "type": "processing_error"
        })

@socketio.on("binary_frame", namespace="/ffai")
def handle_ffai_binary_frame(data, metadata):
    """Handle binary frame data (more efficient than base64)"""
    try:
        # Metadata es dict con timestamp, quality, etc.
        if isinstance(metadata, str):
            metadata = json.loads(metadata)

        # Procesar bytes directamente (si el cliente envía bytes)
        # Por ahora fallback a frame normal
        handle_ffai_frame({
            "imageBase64": data if isinstance(data, str) else "",
            "clientState": metadata
        })

    except Exception as e:
        print(f"❌ Error en binary_frame: {e}")
        emit("error", {"message": str(e)})

@socketio.on("health_check", namespace="/ffai")
def handle_health_check():
    """Health check con acknowledgment"""
    emit("health_response", {
        "status": "ok",
        "timestamp": time.time(),
        "clients": stats["clients_connected"],
        "frames": stats["frames_processed"]
    })

@socketio.on("config_update", namespace="/ffai")
def handle_config_update(config):
    """Actualizar configuración en runtime"""
    try:
        if isinstance(config, str):
            config = json.loads(config)

        print(f"⚙️ Config update recibida: {config}")

        # Aplicar configuración al engine si aplica
        # Por ahora solo confirmar recepción
        emit("config_ack", {
            "status": "updated",
            "config": config
        })

    except Exception as e:
        emit("error", {"message": f"Error updating config: {e}"})

print("✅ API Flask-SocketIO configurada (namespace: /ffai)")
print("   Eventos: connect, disconnect, frame, binary_frame, health_check, config_update")

✅ API Flask-SocketIO configurada (namespace: /ffai)
   Eventos: connect, disconnect, frame, binary_frame, health_check, config_update


## Celda 7: Conectar Ngrok (URL Pública)

In [45]:
from pyngrok import ngrok
import nest_asyncio
import os

# Patch para asyncio
nest_asyncio.apply()

# Token ngrok desde variable de entorno (más seguro)
NGROK_TOKEN = os.environ.get("NGROK_TOKEN", "3CU4gdr4ZZBNCaiS2SS0M65EZZH_55e54r4qkAkq8aEu2MQE8")
ngrok.set_auth_token(NGROK_TOKEN)

# Crear túnel
public_url = ngrok.connect(5000, "http")
ngrok_domain = public_url.public_url.replace("https://", "").replace("http://", "")

print("\n" + "="*60)
print("🌐 URL PÚBLICA DEL SERVIDOR:")
print(f"   {public_url}")
print("="*60)
print(f"\n📋 Configura tu APK:")
print(f"1. Abre la app FFAIAssistant")
print(f"2. En el campo de URL ingresa: {ngrok_domain}")
print(f"3. Presiona 'Conectar'")
print(f"\n📱 Configuración en MainActivity.kt (opcional):")
print(f'   const val SERVER_URL = "{ngrok_domain}"')
print(f'   const val NAMESPACE = "/ffai"')
print(f"\n🔌 SocketIO Namespace: /ffai")
print(f"✅ Servidor listo para conexiones!")


🌐 URL PÚBLICA DEL SERVIDOR:
   NgrokTunnel: "https://poem-tipping-glitter.ngrok-free.dev" -> "http://localhost:5000"

📋 Configura tu APK:
1. Abre la app FFAIAssistant
2. En el campo de URL ingresa: poem-tipping-glitter.ngrok-free.dev
3. Presiona 'Conectar'

📱 Configuración en MainActivity.kt (opcional):
   const val SERVER_URL = "poem-tipping-glitter.ngrok-free.dev"
   const val NAMESPACE = "/ffai"

🔌 SocketIO Namespace: /ffai
✅ Servidor listo para conexiones!


## Celda 8: 🚀 INICIAR SERVIDOR

In [ ]:
print("🚀 Iniciando servidor FFAI...")
print("   Presiona STOP cuando quieras detener\n")

# Iniciar
socketio.run(app, host="0.0.0.0", port=5000, debug=False, allow_unsafe_werkzeug=True)

🚀 Iniciando servidor FFAI...
   Presiona STOP cuando quieras detener

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.28.0.12:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [21/Apr/2026 23:07:39] "GET / HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [21/Apr/2026 23:07:55] "GET / HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [21/Apr/2026 23:08:10] "GET / HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [21/Apr/2026 23:09:37] "GET / HTTP/1.1" 200 -


📱 [/ffai] Cliente conectado: SS2fJXpZg5hvkQwCAAAB
   Total clientes: 1
📊 Frames procesados: 100 | Clientes: 1 | Avg latency: 1.0ms
📊 Frames procesados: 200 | Clientes: 1 | Avg latency: 1.8ms
📊 Frames procesados: 300 | Clientes: 1 | Avg latency: 0.9ms
📊 Frames procesados: 400 | Clientes: 1 | Avg latency: 1.1ms
📊 Frames procesados: 500 | Clientes: 1 | Avg latency: 0.9ms
📊 Frames procesados: 600 | Clientes: 1 | Avg latency: 0.8ms
📊 Frames procesados: 700 | Clientes: 1 | Avg latency: 0.9ms
📊 Frames procesados: 800 | Clientes: 1 | Avg latency: 1.2ms
📊 Frames procesados: 900 | Clientes: 1 | Avg latency: 0.9ms
📊 Frames procesados: 1000 | Clientes: 1 | Avg latency: 0.8ms
📊 Frames procesados: 1100 | Clientes: 1 | Avg latency: 1.0ms
📊 Frames procesados: 1200 | Clientes: 1 | Avg latency: 1.1ms
📊 Frames procesados: 1300 | Clientes: 1 | Avg latency: 1.8ms
📊 Frames procesados: 1400 | Clientes: 1 | Avg latency: 1.1ms
📊 Frames procesados: 1500 | Clientes: 1 | Avg latency: 1.7ms
📊 Frames procesados: 16

INFO:werkzeug:127.0.0.1 - - [21/Apr/2026 23:15:40] "GET / HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [21/Apr/2026 23:15:40] "GET /favicon.ico HTTP/1.1" 404 -


📊 Frames procesados: 7700 | Clientes: 1 | Avg latency: 1.3ms
📊 Frames procesados: 7800 | Clientes: 1 | Avg latency: 0.9ms
📊 Frames procesados: 7900 | Clientes: 1 | Avg latency: 1.2ms
📊 Frames procesados: 8000 | Clientes: 1 | Avg latency: 0.9ms
📊 Frames procesados: 8100 | Clientes: 1 | Avg latency: 1.7ms
📊 Frames procesados: 8200 | Clientes: 1 | Avg latency: 0.9ms
📊 Frames procesados: 8300 | Clientes: 1 | Avg latency: 0.8ms
📊 Frames procesados: 8400 | Clientes: 1 | Avg latency: 2.2ms
📊 Frames procesados: 8500 | Clientes: 1 | Avg latency: 1.0ms
📊 Frames procesados: 8600 | Clientes: 1 | Avg latency: 0.9ms
📊 Frames procesados: 8700 | Clientes: 1 | Avg latency: 1.1ms
📊 Frames procesados: 8800 | Clientes: 1 | Avg latency: 0.9ms
📊 Frames procesados: 8900 | Clientes: 1 | Avg latency: 1.0ms
📱 [/ffai] Cliente conectado: xhkyvj6P8i61boRdAAAD
   Total clientes: 2
📊 Frames procesados: 9000 | Clientes: 2 | Avg latency: 1.6ms
📊 Frames procesados: 9100 | Clientes: 2 | Avg latency: 1.3ms


...

INFO:werkzeug:127.0.0.1 - - [21/Apr/2026 23:27:55] "GET /socket.io/?EIO=4&transport=websocket HTTP/1.1" 500 -
ERROR:werkzeug:Error on request:
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/werkzeug/serving.py", line 371, in run_wsgi
    execute(self.server.app)
  File "/usr/local/lib/python3.12/dist-packages/werkzeug/serving.py", line 337, in execute
    write(b"")
  File "/usr/local/lib/python3.12/dist-packages/werkzeug/serving.py", line 262, in write
    assert status_set is not None, "write() before start_response"
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: write() before start_response


📴 [/ffai] Cliente desconectado: bpmMdMa4OTlZnCXZAAAT
📱 [/ffai] Cliente conectado: FGh_rGZYlDoNxI0TAAAV
   Total clientes: 1


INFO:werkzeug:127.0.0.1 - - [21/Apr/2026 23:28:23] "GET / HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [21/Apr/2026 23:28:33] "GET /socket.io/?EIO=4&transport=websocket HTTP/1.1" 500 -
ERROR:werkzeug:Error on request:
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/werkzeug/serving.py", line 371, in run_wsgi
    execute(self.server.app)
  File "/usr/local/lib/python3.12/dist-packages/werkzeug/serving.py", line 337, in execute
    write(b"")
  File "/usr/local/lib/python3.12/dist-packages/werkzeug/serving.py", line 262, in write
    assert status_set is not None, "write() before start_response"
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: write() before start_response


📴 [/ffai] Cliente desconectado: FGh_rGZYlDoNxI0TAAAV


INFO:werkzeug:127.0.0.1 - - [21/Apr/2026 23:28:43] "GET / HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [21/Apr/2026 23:28:45] "GET / HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [21/Apr/2026 23:28:49] "GET / HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [21/Apr/2026 23:28:57] "GET / HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [21/Apr/2026 23:29:12] "GET / HTTP/1.1" 200 -


## Celda 9: 🎓 Training Pipeline - Model Training & Distillation

Celdas adicionales para entrenamiento, destilación y exportación a TFLite.
Ejecutar cuando tengas suficientes experiencias recolectadas.

In [ ]:
# Celda 10: Training - Perception CNN (Teacher Model)

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np

class PerceptionCNN(nn.Module):
    """Teacher CNN - Modelo grande para detección de enemigos/HUD"""
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 12 * 20, 256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256, 128),
        )
        # Outputs: heatmap 20x12 + HUD 4 valores
        self.heatmap_head = nn.Linear(128, 240)  # 20x12 grid
        self.hud_head = nn.Linear(128, 4)         # health, ammo, coins, armor

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        heatmap = torch.sigmoid(self.heatmap_head(x))
        hud = torch.sigmoid(self.hud_head(x))
        return heatmap, hud

print("✅ PerceptionCNN (Teacher) definida")
print("   Input: 160x96x3 RGB")
print("   Output: heatmap 20x12 + HUD 4 valores")

# Instanciar modelo teacher
teacher_model = PerceptionCNN().to(device)
teacher_params = sum(p.numel() for p in teacher_model.parameters())
print(f"📊 Parámetros teacher: {teacher_params:,} (~{teacher_params*4/1024/1024:.1f}MB fp32)")

✅ Checkpoint guardado: /content/drive/MyDrive/FFAI/checkpoints/manual_save.pth


In [ ]:
# Celda 11: Knowledge Distillation - Student Model

class StudentPerceptionCNN(nn.Module):
    """Student CNN - Modelo pequeño para móvil (distilled from teacher)"""
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 24 * 40, 64), nn.ReLU(),
            nn.Linear(64, 64),
        )
        self.heatmap_head = nn.Linear(64, 240)
        self.hud_head = nn.Linear(64, 4)

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        heatmap = torch.sigmoid(self.heatmap_head(x))
        hud = torch.sigmoid(self.hud_head(x))
        return heatmap, hud

student_model = StudentPerceptionCNN().to(device)
student_params = sum(p.numel() for p in student_model.parameters())
print(f"✅ StudentPerceptionCNN definida")
print(f"   Parámetros student: {student_params:,} (~{student_params*4/1024/1024:.1f}MB fp32)")
print(f"   Compresión: {teacher_params/student_params:.1f}x smaller than teacher")

In [ ]:
# Celda 12: Policy Model - Decision MLP

class PolicyMLP(nn.Module):
    """MLP for tactical decisions (32 inputs → 15 actions)"""
    def __init__(self, input_dim=32, hidden_dim=64, actions=15):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim//2), nn.ReLU(),
            nn.Linear(hidden_dim//2, actions + 4),  # 15 actions + x,y,duration,confidence
        )

    def forward(self, x):
        return self.net(x)

policy_model = PolicyMLP().to(device)
policy_params = sum(p.numel() for p in policy_model.parameters())
print(f"✅ PolicyMLP definida")
print(f"   Input: {32} features (estado del juego)")
print(f"   Output: {15} acciones + 4 parámetros")
print(f"   Parámetros: {policy_params:,} (~{policy_params*4/1024/1024:.1f}MB fp32)")

In [ ]:
# Celda 13: Quantization & TFLite Export

import tempfile
import os

def export_to_tflite(model, input_shape, output_path, representative_dataset=None):
    """Export PyTorch model to quantized TFLite INT8"""

    # 1. Save PyTorch model temporarily
    temp_pt = tempfile.mktemp(suffix='.pt')
    torch.save(model.state_dict(), temp_pt)

    # 2. Create ONNX
    dummy_input = torch.randn(1, *input_shape).to(device)
    temp_onnx = tempfile.mktemp(suffix='.onnx')

    torch.onnx.export(
        model, dummy_input, temp_onnx,
        input_names=['input'],
        output_names=['output'],
        dynamic_axes={'input': {0: 'batch_size'}, 'output': {0: 'batch_size'}},
        opset_version=11
    )

    print(f"✅ ONNX exportado: {os.path.getsize(temp_onnx)/1024:.1f}KB")

    # 3. Convert to TFLite using tflite_converter (if available)
    try:
        import tensorflow as tf

        converter = tf.lite.TFLiteConverter.from_onnx_model(temp_onnx)

        # Quantization settings
        converter.optimizations = [tf.lite.Optimize.DEFAULT]

        if representative_dataset:
            # INT8 quantization with representative dataset
            def rep_data_gen():
                for _ in range(100):
                    yield [np.random.randn(1, *input_shape).astype(np.float32)]
            converter.representative_dataset = rep_data_gen
            converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
            converter.inference_input_type = tf.int8
            converter.inference_output_type = tf.int8

        tflite_model = converter.convert()

        with open(output_path, 'wb') as f:
            f.write(tflite_model)

        print(f"✅ TFLite INT8 exportado: {output_path}")
        print(f"   Tamaño: {os.path.getsize(output_path)/1024:.1f}KB")

    except ImportError:
        print("⚠️ TensorFlow no instalado. Guardando como ONNX.")
        os.rename(temp_onnx, output_path.replace('.tflite', '.onnx'))

# Export models
print("🔄 Exportando modelos...")
export_to_tflite(student_model, (3, 96, 160), "/content/drive/MyDrive/FFAI/models/perception.tflite")
export_to_tflite(policy_model, (32,), "/content/drive/MyDrive/FFAI/models/policy.tflite")

In [ ]:
# Celda 14: Batch Upload Receiver - Recibir experiencias del móvil

import gzip
import json
from datetime import datetime

# Storage for received experiences
received_experiences = []
BATCH_SAVE_DIR = "/content/drive/MyDrive/FFAI/batches"
os.makedirs(BATCH_SAVE_DIR, exist_ok=True)

@app.route("/upload_batch", methods=["POST"])
def upload_batch():
    """Recibe batch de experiencias comprimido desde el móvil"""
    try:
        # Recibir datos comprimidos
        compressed_data = request.data

        # Descomprimir
        json_data = gzip.decompress(compressed_data).decode('utf-8')
        batch = json.loads(json_data)

        # Guardar batch en Drive
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        batch_file = f"{BATCH_SAVE_DIR}/batch_{timestamp}.json.gz"

        with gzip.open(batch_file, 'wt') as f:
            json.dump(batch, f)

        # Agregar a memoria
        received_experiences.extend(batch)

        print(f"✅ Batch recibido: {len(batch)} experiencias")
        print(f"   Total acumulado: {len(received_experiences)} experiencias")
        print(f"   Guardado en: {batch_file}")

        return jsonify({
            "status": "success",
            "experiences_received": len(batch),
            "total_experiences": len(received_experiences)
        })

    except Exception as e:
        print(f"❌ Error recibiendo batch: {e}")
        return jsonify({"status": "error", "message": str(e)}), 500

@app.route("/stats", methods=["GET"])
def get_stats():
    """Estadísticas de entrenamiento"""
    return jsonify({
        "total_experiences": len(received_experiences),
        "batches_saved": len([f for f in os.listdir(BATCH_SAVE_DIR) if f.endswith('.json.gz')]),
        "model_versions": ["v1.0"],
        "last_training": None
    })

print("✅ Endpoints de batch upload añadidos")
print("   POST /upload_batch - Recibe experiencias del móvil")
print("   GET /stats - Estadísticas de entrenamiento")

In [ ]:
# Celda 15: Model Distribution - Descargar modelos TFLite

from flask import send_file, abort

MODELS_DIR = "/content/drive/MyDrive/FFAI/models"
os.makedirs(MODELS_DIR, exist_ok=True)

# Model versions registry
model_versions = {
    "perception": {"v1": f"{MODELS_DIR}/perception.tflite"},
    "policy": {"v1": f"{MODELS_DIR}/policy.tflite"},
    "economy": {"v1": f"{MODELS_DIR}/economy.tflite"},
}

@app.route("/models/<model_name>/<version>", methods=["GET"])
def download_model(model_name, version):
    """Descargar modelo TFLite específico"""
    try:
        if model_name not in model_versions:
            return jsonify({"error": f"Model {model_name} not found"}), 404

        if version not in model_versions[model_name]:
            return jsonify({"error": f"Version {version} not found"}), 404

        model_path = model_versions[model_name][version]

        if not os.path.exists(model_path):
            return jsonify({"error": "Model file not found on server"}), 404

        return send_file(
            model_path,
            mimetype='application/octet-stream',
            as_attachment=True,
            download_name=f"{model_name}_{version}.tflite"
        )

    except Exception as e:
        return jsonify({"error": str(e)}), 500

@app.route("/models/list", methods=["GET"])
def list_models():
    """Listar modelos disponibles"""
    available = {}
    for name, versions in model_versions.items():
        available[name] = {
            "versions": list(versions.keys()),
            "sizes": {v: os.path.getsize(p)/1024 if os.path.exists(p) else 0
                      for v, p in versions.items()}
        }
    return jsonify(available)

@app.route("/models/latest", methods=["GET"])
def get_latest_models():
    """Obtener URLs de los modelos más recientes"""
    # Usar el dominio ngrok actual
    base_url = f"https://{ngrok_domain}"

    return jsonify({
        "perception": {"url": f"{base_url}/models/perception/v1", "version": "v1"},
        "policy": {"url": f"{base_url}/models/policy/v1", "version": "v1"},
        "economy": {"url": f"{base_url}/models/economy/v1", "version": "v1"},
    })

print("✅ Model distribution endpoints añadidos")
print("   GET /models/<name>/<version> - Descargar modelo")
print("   GET /models/list - Listar modelos disponibles")
print("   GET /models/latest - URLs de modelos más recientes")

In [ ]:
# Celda 16: Training Loop - Entrenar con experiencias recolectadas

def train_on_experiences(experiences, model, optimizer, epochs=10):
    """Entrenar modelo con experiencias recolectadas"""
    if not experiences:
        print("⚠️ No hay experiencias para entrenar")
        return

    print(f"🎓 Entrenando con {len(experiences)} experiencias...")

    model.train()
    losses = []

    for epoch in range(epochs):
        total_loss = 0
        count = 0

        for exp in experiences:
            # Parsear experiencia
            state = torch.FloatTensor([
                exp.get('h', 1.0),  # health
                exp.get('a', 1.0),  # ammo
                exp.get('e', 0),    # enemy present
                1.0, 0.5, 0.5, 0.5  # padding
            ]).to(device)

            action_idx = exp.get('a_idx', 0)
            reward = exp.get('r', 0)

            # Forward
            output = model(state)

            # Loss (policy gradient simplificado)
            target = torch.zeros(15).to(device)
            target[action_idx] = reward

            loss = nn.MSELoss()(output[:15], target)

            # Backward
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            count += 1

        avg_loss = total_loss / max(count, 1)
        losses.append(avg_loss)

        if epoch % 2 == 0:
            print(f"   Epoch {epoch}: loss={avg_loss:.4f}")

    print(f"✅ Entrenamiento completo. Final loss: {losses[-1]:.4f}")
    return losses

# Setup optimizer
optimizer = optim.Adam(policy_model.parameters(), lr=0.001)

# Entrenar si hay experiencias
if received_experiences:
    train_on_experiences(received_experiences, policy_model, optimizer, epochs=10)
else:
    print("💡 No hay experiencias aún. El modelo usará pesos aleatorios.")
    print("   Conecta el móvil y juega para recolectar experiencias.")

In [ ]:
# Celda 17: Auto-Deployment - Notificar a móviles de nuevos modelos

from threading import Thread
import time

class ModelAutoDeployer:
    """Monitorea cambios en modelos y notifica a clientes conectados"""

    def __init__(self, socketio, check_interval=60):
        self.socketio = socketio
        self.check_interval = check_interval
        self.running = False
        self.last_modifications = {}

    def start(self):
        self.running = True
        Thread(target=self._monitor, daemon=True).start()
        print("🔄 Auto-deployer iniciado")

    def _monitor(self):
        while self.running:
            try:
                for model_name, versions in model_versions.items():
                    for version, path in versions.items():
                        if os.path.exists(path):
                            current_mtime = os.path.getmtime(path)
                            key = f"{model_name}_{version}"

                            if key in self.last_modifications:
                                if current_mtime > self.last_modifications[key]:
                                    print(f"📦 Modelo actualizado: {model_name} v{version}")
                                    self._notify_clients(model_name, version)

                            self.last_modifications[key] = current_mtime

            except Exception as e:
                print(f"❌ Error en auto-deployer: {e}")

            time.sleep(self.check_interval)

    def _notify_clients(self, model_name, version):
        """Notificar a todos los clientes conectados"""
        self.socketio.emit('model_update', {
            'model': model_name,
            'version': version,
            'url': f"https://{ngrok_domain}/models/{model_name}/{version}",
            'timestamp': time.time()
        }, namespace='/ffai', broadcast=True)

# Iniciar auto-deployer
deployer = ModelAutoDeployer(socketio)
deployer.start()

print("✅ Auto-deployer configurado")
print("   - Monitorea /content/drive/MyDrive/FFAI/models/")
print("   - Notifica a móviles cuando hay nuevos modelos")
print("   - Intervalo de chequeo: 60 segundos")

In [ ]:
# Celda 18: Resumen y Uso del Sistema de Entrenamiento

print("""
╔════════════════════════════════════════════════════════════════╗
║         🎓 FFAI Training Pipeline - Fase 6 (Cloud)             ║
╠════════════════════════════════════════════════════════════════╣

📊 FLUJO DE TRABAJO:

1. RECOLECCIÓN (Móvil → Colab):
   - El móvil juega y recolecta experiencias
   - Batch upload cada 5 minutos o 1000 experiencias
   - POST /upload_batch recibe y guarda en Drive

2. ENTRENAMIENTO (Colab):
   - Teacher CNN: Modelo grande para detección
   - Student CNN: Destilado del teacher (~2MB)
   - Policy MLP: Decisiones tácticas (~1MB)
   - Celda 16: Ejecutar training loop

3. EXPORTACIÓN (PyTorch → TFLite):
   - Quantization INT8 para reducir tamaño 4x
   - Export a /content/drive/MyDrive/FFAI/models/
   - Celda 13: Exportar a TFLite

4. DISTRIBUCIÓN (Colab → Móvil):
   - Auto-deployer notifica a móviles
   - GET /models/latest lista URLs
   - Móvil descarga y actualiza modelos

═══════════════════════════════════════════════════════════════

🚀 COMANDOS RÁPIDOS:

# Ver estadísticas:
!curl https://TU_NGROK_URL/stats

# Listar modelos:
!curl https://TU_NGROK_URL/models/list

# Descargar modelo:
!curl -o perception.tflite https://TU_NGROK_URL/models/perception/v1

═══════════════════════════════════════════════════════════════

📈 MONITOREO:
""")

# Mostrar estado actual
print(f"   Experiencias recibidas: {len(received_experiences)}")
print(f"   Batches guardados: {len([f for f in os.listdir(BATCH_SAVE_DIR) if f.endswith('.json.gz')])}")
print(f"   Modelos disponibles: {len([f for f in os.listdir(MODELS_DIR) if f.endswith('.tflite')])}")
print(f"   URL pública: https://{ngrok_domain}")

print("""
═══════════════════════════════════════════════════════════════

💡 SIGUIENTES PASOS:

1. Conecta tu móvil y juega una partida
2. Verifica que lleguen experiencias (GET /stats)
3. Ejecuta Celda 16 para entrenar
4. Ejecuta Celda 13 para exportar a TFLite
5. El móvil descargará automáticamente los nuevos modelos

═══════════════════════════════════════════════════════════════
""")

In [ ]:
## Celda 19: 🚀 INICIAR SERVIDOR (con endpoints de entrenamiento)

print("🚀 Iniciando servidor FFAI con training pipeline...")
print("   - SocketIO: /ffai (acciones en tiempo real)")
print("   - HTTP API: /upload_batch (experiencias)")
print("   - HTTP API: /models/* (distribución de modelos)")
print("   - Auto-deployer: notifica nuevos modelos")
print("\n   Presiona STOP cuando quieras detener\n")

# Iniciar
socketio.run(app, host="0.0.0.0", port=5000, debug=False, allow_unsafe_werkzeug=True)

In [ ]:
## Bonus: Guardar Checkpoint y Exportar Modelos

Ejecuta esto para guardar manualmente los modelos entrenados:

# Guardar PyTorch checkpoint
checkpoint_path = "/content/drive/MyDrive/FFAI/checkpoints/trained_model.pth"
torch.save({
    'teacher_state': teacher_model.state_dict() if 'teacher_model' in globals() else None,
    'student_state': student_model.state_dict() if 'student_model' in globals() else None,
    'policy_state': policy_model.state_dict() if 'policy_model' in globals() else None,
    'optimizer_state': optimizer.state_dict() if 'optimizer' in globals() else None,
    'experiences_count': len(received_experiences),
    'timestamp': time.time()
}, checkpoint_path)

print(f"✅ Checkpoint guardado: {checkpoint_path}")

# Exportar a TFLite si aún no lo has hecho
if os.path.exists("/content/drive/MyDrive/FFAI/models/perception.tflite"):
    size_mb = os.path.getsize("/content/drive/MyDrive/FFAI/models/perception.tflite") / 1024 / 1024
    print(f"📦 perception.tflite: {size_mb:.2f}MB")
else:
    print("⚠️ No hay modelos TFLite exportados. Ejecuta Celda 13.")

# Verificar que los endpoints están configurados
print("\n📡 Endpoints disponibles:")
print("   POST /upload_batch - Recibir experiencias")
print("   GET  /models/list - Listar modelos")
print("   GET  /models/latest - URLs de descarga")
print("   GET  /stats - Estadísticas")